In [ ]:
import os
import zipfile
import pandas as pd
import re
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# === CONFIGURATION ===
ZIP_FILE = "Jammu And Kashmir_All_Live.zip"
EXTRACT_DIR = "Jammu And Kashmir_All_Live"
SUMMARY_FILE = "BOQ_All_Combined_Summary.xlsx"

# === Helper Functions ===
def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value

def extract_dia(text):
    text = str(text).lower()
    match = re.search(r'(\d+)\s*mm', text)
    if match:
        dia = int(match.group(1))
        if dia in [100, 200, 250, 3500, 1700]:  # optional filter
            return dia
    return None

def find_column(columns, keywords):
    for col in columns:
        if any(k in str(col).lower() for k in keywords):
            return col
    return None

# === STEP 1: Extract ZIP ===
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)
print(f"✅ Extracted ZIP to: {EXTRACT_DIR}")

# === STEP 2: Process Files and Collect Combined Data ===
combined_rows = []

def process_boq_file(file_path, tdr_name, file_name):
    try:
        df_raw = pd.read_excel(file_path, header=None)
    except Exception as e:
        print(f"❌ Cannot read {file_name}: {e}")
        return

    header_row_index = None
    for i in range(min(30, len(df_raw))):
        row = df_raw.iloc[i].astype(str).str.lower()
        if any("desc" in cell or "item" in cell for cell in row) and any("unit" in cell for cell in row):
            header_row_index = i
            break

    if header_row_index is None:
        print(f"⚠️ Header not found in {file_name}")
        return

    try:
        df = pd.read_excel(file_path, header=header_row_index)
    except Exception as e:
        print(f"❌ Failed to load structured data from {file_name}: {e}")
        return

    desc_col = find_column(df.columns, ['desc', 'item'])
    unit_col = find_column(df.columns, ['unit'])
    qty_col = find_column(df.columns, ['qty', 'quantity'])
    rate_col = find_column(df.columns, ['rate', 'amount', 'estimate'])

    if not desc_col or not unit_col:
        print(f"⚠️ Skipping {file_name} due to missing columns")
        return

    df[unit_col] = df[unit_col].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)

    def is_pipe_row(row):
        desc = str(row[desc_col]).lower()
        return any(x in desc for x in ["pipe", "di", "ci", "m.s", "k-7", "k-9", "hdpe", "pvc", "upvc"])

    filtered_df = df[df.apply(is_pipe_row, axis=1)].copy()
    if filtered_df.empty:
        print(f"ℹ️ No matching rows in {file_name}")
        return

    filtered_df["Item Description"] = filtered_df[desc_col].astype(str)
    filtered_df["DIA"] = filtered_df["Item Description"].apply(extract_dia)
    filtered_df["estimate rate"] = pd.to_numeric(filtered_df[rate_col], errors='coerce') if rate_col else 0
    filtered_df["Units"] = df[unit_col].astype(str).str.strip()
    filtered_df["Quantity"] = pd.to_numeric(df[qty_col], errors='coerce') if qty_col else None

    # Add new columns for K-9 and K-7 flags
    filtered_df["K-9"] = filtered_df["Item Description"].str.lower().str.contains("k-9|k9").map({True: "Yes", False: ""})
    filtered_df["K-7"] = filtered_df["Item Description"].str.lower().str.contains("k-7").map({True: "Yes", False: ""})

    filtered_df["TDR Folder"] = tdr_name
    filtered_df["BOQ File"] = file_name

    final_df = filtered_df[[
        "TDR Folder", "BOQ File", "Item Description", "K-9", "K-7",
        "DIA", "estimate rate", "Units", "Quantity"
    ]].copy()

    combined_rows.extend(final_df.to_dict(orient="records"))

# === STEP 3: Walk All BOQ Files ===
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith(('.xls', '.xlsx')) and not file.startswith('~$') and '_filtered' not in file.lower():
            full_path = os.path.join(root, file)
            tdr_folder = os.path.basename(os.path.dirname(full_path))
            process_boq_file(full_path, tdr_folder, file)

# === STEP 4: Save Combined Summary ===
if combined_rows:
    df_summary = pd.DataFrame(combined_rows)
    df_summary.insert(0, "SL No", range(1, len(df_summary) + 1))
    df_summary = df_summary.astype(str).map(clean_illegal_chars)
    df_summary.to_excel(SUMMARY_FILE, index=False, engine="openpyxl")
    print(f"\n✅ Combined BOQ summary saved to: {SUMMARY_FILE}")
    print(f"🧾 Total entries: {len(df_summary)}")
else:
    print("⚠️ No matching BOQ rows found.")
